# Stochastic Calculus, Step by Step · Lesson 16: Itô's lemma

Companion notebook for the lesson. Every number in the lesson's text, tables and figures is produced by running this notebook and read from `build/outputs.json`.

> **The course's numbers.** A share priced at S₀ = 100, a strike K = 105, a risk-free rate r = 3% (continuously compounded), a volatility σ = 20% and a horizon T = 1 year. Made up for teaching; every random number is simulated with a fixed seed.
>
> Educational material only, not financial advice.

1. Setup · 2. Itô processes · 3. Where the extra term comes from · 4. Itô's lemma on one path · 5. Functions of W · 6. The input's volatility moves the output's drift · 7. A moment by Itô: E[W⁴] = 3T² · 8. Exercises · 9. Export

**Running in Google Colab?** Nothing to install: this notebook only needs numpy, pandas and matplotlib, which Colab already has. Run the cells in order.

In [ ]:
# Parameters (papermill overrides these)
output_json = "build/outputs.json"
seed = 2026

## 1. Setup

In [ ]:
import json
import math
import datetime as dt
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

S0, K, r, sigma, T = 100.0, 105.0, 0.03, 0.20, 1.0     # the course's numbers
N = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))   # standard normal distribution function
out = {"meta": {
    "series": "Stochastic Calculus, Step by Step",
    "lesson": 16,
    "quantlib_version": "not used",
    "quotes_label": "simulated (no market data)",
    "seed": seed,
    "generated_at": dt.datetime.now().isoformat(timespec="seconds"),
}, "inputs": {"S0": S0, "K": K, "r": r, "sigma": sigma, "T": T, "r_pct": r * 100, "sigma_pct": sigma * 100}}

## 2. Itô processes

An **Itô process** moves by a drift and a random part: $dX_t = a_t\,dt + b_t\,dW_t$, short for $X_t = X_0 + \int_0^ta_s\,ds + \int_0^tb_s\,dW_s$. With constant $a = 0.03$ and $b = 0.2$ it is $X_t = 0.03t + 0.2W_t$: mean $0.03t$, variance $0.04t$.

In [ ]:
rng = np.random.default_rng(seed)
a_, b_ = r, sigma
Xt = a_ * 1 + b_ * rng.normal(0, 1, 200_000)
out["itoproc"] = {"a": a_, "b": b_, "mean": float(Xt.mean()), "var": float(Xt.var()), "var_theory": b_**2}
out["itoproc"]

## 3. Where the extra term comes from

Taylor (Lesson 1) to second order: $f(x + \Delta x) - f(x) = f'(x)\Delta x + \tfrac12f''(x)\Delta x^2 + \dots$. For a smooth path, $\Delta x^2$ is of order $\Delta t^2$ and the sum of those terms vanishes. For Brownian motion, $\Delta W^2$ is of order $\Delta t$, and the multiplication table of Lesson 14 says their sum is $\sum\Delta t = T$: **the second-order term survives**.

The picture: a symmetric move $\pm h$ around $x$ raises the **average** of a convex $f$ by about $\tfrac12f''(x)h^2$.

In [ ]:
x0, h = 1.0, 0.5
f = lambda x: x**2
xs = np.linspace(0, 2, 201)
out["convex"] = {"x0": x0, "h": h, "f0": f(x0), "avg": (f(x0 - h) + f(x0 + h)) / 2, "half_f2_h2": 0.5 * 2 * h**2,
                 "fig": {"x": xs.tolist(), "f": f(xs).tolist(), "chord_x": [x0 - h, x0 + h],
                         "chord_y": [f(x0 - h), f(x0 + h)],
                         "ticks": [{"x": x, "label": f"{x:g}"} for x in (0, 0.5, 1, 1.5, 2)]}}
x2 = np.linspace(0.4, 1.6, 121)
out["convex"]["fig"].update({"x2": x2.tolist(), "f2": f(x2).tolist(), "chord2": (f(x0 - h) + 2 * x0 * (x2 - x0 + h)).tolist()})
out["convex"]["fig"]["ticks2"] = [{"x": x, "label": f"{x:g}"} for x in (0.5, 1, 1.5)]
out["convex"]["fig"]["pts_x"] = [x0 - h, x0, x0, x0 + h]
out["convex"]["fig"]["pts_y"] = [f(x0 - h), f(x0), (f(x0 - h) + f(x0 + h)) / 2, f(x0 + h)]
{k_: v for k_, v in out["convex"].items() if k_ != "fig"}

## 4. Itô's lemma on one path

For $f(W)$: $\;df(W_t) = f'(W_t)\,dW_t + \tfrac12f''(W_t)\,dt$. Check it on one path, for $f(x) = x^2$ and $f(x) = e^x$: the change $f(W_1) - f(0)$ against the first-order sum $\sum f'(W_{t_i})\Delta W_i$ alone, and with the Itô term $\sum\tfrac12f''(W_{t_i})\Delta t$ added.

In [ ]:
fine_n = 2**18
W_f = np.concatenate([[0], np.cumsum(rng.normal(0, math.sqrt(1 / fine_n), fine_n))])
funcs = {"x^2": (lambda x: x**2, lambda x: 2 * x, lambda x: 2 + 0 * x),
         "e^x": (np.exp, np.exp, np.exp)}
path_rows = []
for name, (f_, f1, f2) in funcs.items():
    change = float(f_(W_f[-1]) - f_(0.0))
    for k_ in (6, 10, 14, 18):
        n = 2**k_
        Wn = W_f[:: fine_n // n]; d_ = np.diff(Wn); dt_ = 1 / n
        first = float((f1(Wn[:-1]) * d_).sum())
        ito = float((0.5 * f2(Wn[:-1]) * dt_).sum())
        path_rows.append({"f": name, "n": n, "change": change, "first": first, "with_ito": first + ito,
                          "gap_first": change - first, "gap_ito": change - first - ito})
out["onepath"] = {"rows": path_rows, "W_1": float(W_f[-1])}
pd.DataFrame(path_rows).round(4)

## 5. Functions of W

$d(W^2) = 2W\,dW + dt$, $\;d(W^3) = 3W^2\,dW + 3W\,dt$, $\;d(e^W) = e^W\,dW + \tfrac12e^W\,dt$. Taking averages kills the $dW$ terms (Lesson 15: Itô integrals have mean 0). For $m(t) = E[e^{W_t}]$ this gives $m'(t) = \tfrac12m(t)$, so $m(t) = e^{t/2}$, the lognormal mean of Lesson 4, found without integrating a density.

In [ ]:
Z1 = rng.normal(0, 1, 400_000)
out["funcs"] = {"E_eW": {"t" + f"{t:g}".replace(".", ""): float(np.exp(math.sqrt(t) * Z1).mean()) for t in (0.5, 1.0, 2.0)},
                "E_eW_theory": {"t" + f"{t:g}".replace(".", ""): math.exp(t / 2) for t in (0.5, 1.0, 2.0)},
                "E_W2": float((Z1**2).mean())}
out["funcs"]

## 6. The input's volatility moves the output's drift

For an Itô process $dX = a\,dt + b\,dW$ and a smooth $f(t, x)$:

$$df(t, X_t) = \Big(f_t + a\,f_x + \tfrac12b^2f_{xx}\Big)dt + b\,f_x\,dW_t .$$

The output's drift depends on the input's **volatility** $b$, through $\tfrac12b^2f_{xx}$. Example: the log price $X = (r - \tfrac12\sigma^2)t + \sigma W$ has drift $r - \tfrac12\sigma^2$, but the price $S = S_0e^X$ has drift $rS$: the exponential (convex) adds $\tfrac12\sigma^2$ back.

In [ ]:
n_paths, n_steps = 50_000, 250
dt = 1 / n_steps
dW = rng.normal(0, math.sqrt(dt), (n_paths, n_steps))
X = np.cumsum((r - 0.5 * sigma**2) * dt + sigma * dW, axis=1)
S_ = S0 * np.exp(X)
checks = [0.25, 0.5, 1.0]
out["inout"] = {"rows": [{"t": t, "EX": float(X[:, int(t * n_steps) - 1].mean()), "EX_theory": (r - 0.5 * sigma**2) * t,
                          "ES": float(S_[:, int(t * n_steps) - 1].mean()), "ES_theory": S0 * math.exp(r * t)} for t in checks],
                "drift_in": r - 0.5 * sigma**2, "drift_out": r, "correction": 0.5 * sigma**2}
pd.DataFrame(out["inout"]["rows"])

## 7. A moment by Itô: E[W⁴] = 3T²

$d(W^4) = 4W^3\,dW + 6W^2\,dt$. Average: $\frac{d}{dt}E[W_t^4] = 6E[W_t^2] = 6t$, so $E[W_T^4] = 3T^2$. That is $E[Z^4] = 3$ from Lesson 4, rederived with no integral of a density.

In [ ]:
out["moment"] = {"rows": [{"T": T_, "sim": float(((math.sqrt(T_) * Z1) ** 4).mean()), "theory": 3 * T_**2}
                          for T_ in (0.5, 1.0, 2.0)]}
pd.DataFrame(out["moment"]["rows"])

## 8. Exercises

Try each one before looking anything up. Full solutions are in the companion volume, *Part 3: Solutions*.

**Exercise 1.** Use Itô's lemma to find $d(W_t^3)$ and $d(e^{W_t})$. Check each against $f(W_T) - f(0) = \int f'\,dW + \int\tfrac12f''\,dt$ by taking expectations.

**Exercise 2.** Use the two-variable form to find $d(tW_t)$. Which term would ordinary calculus give, and is there an Itô correction here?

**Exercise 3.** Show that $Z_t = e^{\sigma W_t - \frac12\sigma^2t}$ satisfies $dZ_t = \sigma Z_t\,dW_t$: it has no $dt$ term.

**Exercise 4.** Use Itô's lemma on $W^6$ to show $E[W_T^6] = 15T^3$.

**Exercise 5.** Let $dX = a\,dt + b\,dW$ with constants $a$, $b$ and $X_0 = 0$. Find $d(X^2)$, and from it $E[X_T^2]$. Check with $X_T = aT + bW_T$.

**Exercise 6.** Let $Y_t = \cos W_t$. Find $dY_t$, take expectations and solve the resulting equation to show $E[\cos W_t] = e^{-t/2}$.

**Exercise 7.** (Code) On one Brownian path, check Itô's lemma for $f(x) = x^3$: compare $W_1^3$ with $\sum 3W_{t_i}^2\Delta W_i + \sum 3W_{t_i}\Delta t$ for $2^k$ steps, $k = 6, \dots, 18$.

**Exercise 8.** (Code) Check $E[\cos W_t] = e^{-t/2}$ and $E[W_t^4] = 3t^2$ by simulation for $t = 0.5, 1, 2$.

## 9. Export

In [ ]:
path = Path(output_json)
path.parent.mkdir(parents=True, exist_ok=True)
path.write_text(json.dumps(out, indent=2, default=float))
print("wrote", path.resolve(), f"({path.stat().st_size / 1024:.0f} KB)")